In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
from pathlib import Path
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
#
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

benchmark_name = "billionaires-statistics-2023"
data = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "Billionaires Statistics Dataset.csv")
factor = 800
data = pd.concat([data] * factor, ignore_index=True)
data.info()

In [ ]:
### cell 1 ###

data.head(5)

In [ ]:
### cell 2 ###

data.describe()

In [ ]:
### cell 3 ###

# Use value_counts() directly so that the tie‐breaking and ordering exactly match the original output.
# With %load_ext cudf.pandas loaded, this will run on the GPU under the hood.
country_names = data["country"].value_counts()

In [ ]:
### cell 4 ###

country_names

In [ ]:
### cell 5 ###

data_100 = data.loc[:100, ['finalWorth', 'category', 'country']]

In [ ]:
### cell 6 ###

data_100_category = data_100["category"].value_counts()

In [ ]:
### cell 7 ###
# use GPU‐accelerated ‘isin’ instead of a Python scalar comparison
data_usa = data[data["country"].isin(["United States"]) ]

In [ ]:
### cell 8 ###

data_usa_category = data_usa["category"].value_counts()
data_usa_category

In [ ]:
### cell 9 ###

data_usa_city = data_usa["city"].value_counts()
print(data_usa_city)
data_usa_city.info()

In [ ]:
### cell 10 ###

data.head(5)

In [ ]:
### cell 11 ###

category_list = list(data.category.unique())
category_list

In [ ]:
### cell 12 ###

data.finalWorth = data.finalWorth.astype(float)
# compute average finalWorth per category on the GPU
data2 = data.groupby('category').finalWorth.mean().reset_index()
# rename columns to match original
data2 = data2.rename(columns={'category': 'category_list', 'finalWorth': 'worth_average'})
# sort by worth_average descending
new_data = data2.sort_values('worth_average', ascending=False)

In [ ]:
### cell 13 ###

# Optimized for GPU: use default dropna without subset to avoid CPU fallback
data3 = data.dropna()

In [ ]:
### cell 14 ###

data3.head()

In [ ]:
### cell 15 ###

data3.info()  # in the new df we have 238 rows for each column

In [ ]:
### cell 16 ###

# Replace pandas unique (CPU) with cudf drop_duplicates (GPU)
data3.countryOfCitizenship.drop_duplicates()